# **Joint single-cell DNA-RNA Data Processing**

## Description of Notebook

Fill in later.

# 0. Import Packages, Install Dependencies

In [55]:
# Import Initial Packages
import os
import sys
import subprocess
import glob

import pandas as pd
import numpy as np
print(pd.__version__)   # should be ≥2.3 and <3.0
print(np.__version__)   # should be ≥1.26 and <2.1

2.2.2
2.0.2


In [ ]:
# Install Dependencies
!pip install snapatac2-scooby

# Installing collected packages: texttable, cykhash, rustworkx, logistro, igraph, array-api-compat, pytest-timeout, pyfaidx, 
# # choreographer, kaleido, hmmlearn, anndata, macs3, snapatac2-scooby


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 11.3 MB/s eta 0:00:00a 0:00:01


: 

In [ ]:
!pip install -q alphagenome scvi-tools scanpy anndata pysam pyranges scrublet gdown

In [ ]:
!pip install --upgrade google-cloud-bigquery



E: Unable to locate package entrez-direct
/bin/bash: line 1: fasterq-dump: command not found
/bin/bash: line 1: esearch: command not found


In [11]:
%%bash
set -e

# Download and install Entrez Direct into $HOME/edirect
sh -c "$(curl -fsSL https://ftp.ncbi.nlm.nih.gov/entrez/entrezdirect/install-edirect.sh)"

# Show where it was installed
# ls -l $HOME/edirect


Entrez Direct has been successfully downloaded and installed.

In order to complete the configuration process, please execute the following:

  echo "export PATH=/root/edirect:\${PATH}" >> ${HOME}/.bashrc

or manually edit the PATH variable assignment in your .bashrc file.

Would you like to do that automatically now? [y/N]
Holding off, then.

To activate EDirect for this terminal session, please execute the following:

export PATH=${HOME}/edirect:${PATH}



In [12]:
# Add Entrez Direct to PATH for Python Kernel
os.environ["PATH"] = os.path.join(os.environ["HOME"], "edirect") + ":" + os.environ["PATH"]
print(os.environ["PATH"])

/root/edirect:/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin


In [ ]:
# Install NCBI Entrez Direct utilities
#!apt-get install -y -qq entrez-direct 

# Verify installation
#!fasterq-dump --version
print(os.environ["PATH"])

!esearch -help | head -5

/root/edirect:/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
esearch 25.3

Query Specification

  -db            Database name


In [ ]:
%%bash
# Directly install SRA Toolkit from NCBI Server
set -e

cd $HOME
# Get latest Linux 64‑bit build (adjust URL if you need a different platform)
curl -L -o sratoolkit.tar.gz \
  https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz

tar -xzf sratoolkit.tar.gz
ls

edirect
sratoolkit.3.4.1-ubuntu64
sratoolkit.tar.gz


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 85.0M  100 85.0M    0     0   134M      0 --:--:-- --:--:-- --:--:--  134M


In [ ]:
# Add SRA Toolkit to PATH variable

home = os.environ["HOME"]
# Find the extracted sratoolkit directory
toolkits = glob.glob(os.path.join(home, "sratoolkit.*-ubuntu64"))
assert toolkits, "No sratoolkit directory found"
sra_dir = os.path.join(toolkits[0], "bin")

os.environ["PATH"] = sra_dir + ":" + os.environ["PATH"]
print(os.environ["PATH"])

/root/sratoolkit.3.4.1-ubuntu64/bin:/root/edirect:/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin


In [21]:
# Verify installation
!fasterq-dump --version

fasterq-dump : 3.4.1



In [ ]:

#import anndata as ad
#import scanpy as sc



complete


In [56]:
# CONNECT TO GOOGLE DRIVE
import google.colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load files locally from Google Drive (optional)

In [ ]:
# Load files from Google Cloud BigQuery

# Authenticate Session
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

# Set up GCP Project
#project_id = 'bmi-702-project' # Replace with your actual project ID
#from google.cloud import bigquery
#client = bigquery.Client(project=project_id)

Authenticated


In [28]:
print(os.getcwd)

<built-in function getcwd>


# 1. Collect GEO GSE185269 Files

In [ ]:
# Folder and file paths
folder_path = '/content/drive/MyDrive/MIT HST 506/project_data'
GEO_metadata_path_1 = '/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files/GSE185269-GPL18573_series_matrix_edited.txt'
GEO_metadata_path_2 = '/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files/GSE185269-GPL28038_series_matrix_edited.txt'

GEO_series_matrix_1 = (pd.read_table("/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files/GSE185269-GPL18573_series_matrix_edited.txt")).transpose()
GEO_series_matrix_2 = (pd.read_csv(GEO_metadata_path_2)).transpose()

In [ ]:

# Make first row the column names
GEO_series_matrix_1.columns = GEO_series_matrix_1.iloc[0]
GEO_series_matrix_1 = GEO_series_matrix_1[1:].reset_index(drop=True)

GEO_series_matrix_2.columns = GEO_series_matrix_2.iloc[0]
GEO_series_matrix_2 = GEO_series_matrix_2[1:].reset_index(drop=True)

In [82]:
#GEO_series_matrix_1 # 431 rows × 48 columns

#GEO_series_matrix_2 #380 rows × 46 columns



In [84]:
#GEO_series_matrix_2.head()

In [ ]:
# Download processed data (countmatrices) from GEO Project
# Claude suggested code

# Download GEO supplementary files directly via FTP
geo_base = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE185nnn/GSE185269/suppl/"

# List available supplementary files
!curl -s {geo_base} | grep -oP '(?<=href=")[^"]+\.gz'

# Download Specific Files
!wget -P data/processed/ {geo_base}<FILENAME>.gz

# 2. Collect the Run Metadata for SRP339960

In [30]:
items = os.listdir('/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files')
print(items)

['SRP339960_runinfo_original.csv', '.DS_Store']


In [ ]:
# Folder and file paths
folder_path = '/content/drive/MyDrive/MIT HST 506/project_data'
SRA_metadata_path = '/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv'

SRA_metadata = pd.read_csv(SRA_metadata_path)

In [24]:
(SRA_metadata).head()

,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260


In [27]:
len(SRA_metadata['BioSample'].unique())


621

In [ ]:
# Fetch the Run Metadata for SRP339960 study (method using Entrez commands)

!esearch -db sra -query SRP339960 | efetch -format runinfo > /content/drive/MyDrive/"MIT HST 506"/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv


In [33]:
# Load and inspect

runinfo = pd.read_csv("/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv")
print(runinfo.shape)
print(runinfo.columns.tolist())
runinfo.head(10)

(621, 47)
['Run', 'ReleaseDate', 'LoadDate', 'spots', 'bases', 'spots_with_mates', 'avgLength', 'size_MB', 'AssemblyName', 'download_path', 'Experiment', 'LibraryName', 'LibraryStrategy', 'LibrarySelection', 'LibrarySource', 'LibraryLayout', 'InsertSize', 'InsertDev', 'Platform', 'Model', 'SRAStudy', 'BioProject', 'Study_Pubmed_id', 'ProjectID', 'Sample', 'BioSample', 'SampleType', 'TaxID', 'ScientificName', 'SampleName', 'g1k_pop_code', 'source', 'g1k_analysis_group', 'Subject_ID', 'Sex', 'Disease', 'Tumor', 'Affection_Status', 'Analyte_Type', 'Histological_Type', 'Body_Site', 'CenterName', 'Submission', 'dbgap_study_accession', 'Consent', 'RunHash', 'ReadHash']


,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260
5,SRR16195155,2022-12-31 00:17:52,2021-10-04 17:05:48,1249873,179028128,1249873,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,A030D8031AD0F81019F804F0DDBFC276,48036FDD9DC94429BC9BDA44B2A4ED3C
6,SRR16195154,2022-12-31 00:17:52,2021-10-04 17:05:45,1229207,176079810,1229207,143,70,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,E475C90208DE4129E051CF73D9EAFA98,2A7F69DBCE6B811F488FD7450B0D53C6
7,SRR16195153,2022-12-31 00:17:52,2021-10-04 17:05:41,1024753,146792703,1024753,143,59,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,67DA2996604182344AD26D7EE1AF14B5,5FAA152A579A111C612A94EFD9F3606A
8,SRR16195152,2022-12-31 00:17:52,2021-10-04 17:05:36,1015576,145446512,1015576,143,58,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,CF0AEF250FF730207A569BF366135B43,0C2F1867F5C907379937020DB94ABDFA
9,SRR16195151,2022-12-31 00:17:52,2021-10-04 17:05:36,713817,102235095,713817,143,41,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,263EBD86D10A955037FE225C970E5AB5,E72940FD9479ABFBDC9A96D6E4B8E0F2


# 3. Separate scDNA and scRNA Runs

In [37]:
# Separate WGS (DNA) and RNA-seq runs
dna_runs = runinfo[runinfo['LibraryStrategy'] == 'OTHER']['Run'].tolist()
rna_runs = runinfo[runinfo['LibraryStrategy'] == 'RNA-Seq']['Run'].tolist()

print(f"DNA (WGS) runs: {len(dna_runs)}")
print(f"RNA-seq runs:   {len(rna_runs)}")
print("\nDNA runs:", dna_runs[:5])
print("RNA runs:", rna_runs[:5])

DNA (WGS) runs: 525
RNA-seq runs:   96

DNA runs: ['SRR16195157', 'SRR16195156', 'SRR16195155', 'SRR16195154', 'SRR16195153']
RNA runs: ['SRR16195160', 'SRR16195159', 'SRR16195158', 'SRR16195109', 'SRR16195108']


In [45]:
# Sample Name, Library Name, BioSample
print(len((runinfo['SampleName']).unique())) #621
print(len((runinfo['LibraryName']).unique())) #621
print(len((runinfo['BioSample']).unique())) #621
print(len((runinfo['Experiment']).unique())) #621
print(len((runinfo['Sample']).unique())) #621

621
621
621
621
621


In [51]:
rna = runinfo[runinfo["LibraryStrategy"].str.lower() == "rna-seq"]
dna = runinfo[runinfo["LibraryStrategy"].str.lower() != "rna-seq"]

rna[["Run", "LibraryName", "SampleName"]].head(10)


,Run,LibraryName,SampleName
0,SRR16195160,GSM5609425,GSM5609425
1,SRR16195159,GSM5609426,GSM5609426
2,SRR16195158,GSM5609427,GSM5609427
51,SRR16195109,GSM5609428,GSM5609428
52,SRR16195108,GSM5609429,GSM5609429
53,SRR16195107,GSM5609430,GSM5609430
54,SRR16195106,GSM5609431,GSM5609431
55,SRR16195105,GSM5609432,GSM5609432
56,SRR16195104,GSM5609433,GSM5609433
57,SRR16195103,GSM5609434,GSM5609434


In [52]:
dna[["Run", "LibraryName", "SampleName"]].head(10)

,Run,LibraryName,SampleName
3,SRR16195157,GSM5609620,GSM5609620
4,SRR16195156,GSM5609621,GSM5609621
5,SRR16195155,GSM5609622,GSM5609622
6,SRR16195154,GSM5609623,GSM5609623
7,SRR16195153,GSM5609624,GSM5609624
8,SRR16195152,GSM5609625,GSM5609625
9,SRR16195151,GSM5609626,GSM5609626
10,SRR16195150,GSM5609627,GSM5609627
11,SRR16195149,GSM5609628,GSM5609628
12,SRR16195148,GSM5609629,GSM5609629


# 4. Download FASTQ Files

In [86]:
# Define base path on Google Drive
base_path = "/content/drive/MyDrive/MIT HST 506/project_data/FASTQ_files"

os.makedirs(f"{base_path}/dna", exist_ok=True)
os.makedirs(f"{base_path}/rna", exist_ok=True)

# Test with first DNA run
test_dna = dna_runs[0]
!fasterq-dump {test_dna} --split-files --outdir "{base_path}/dna" --threads 4
!gzip "{base_path}/dna/{test_dna}"*.fastq

# Test with first RNA run
test_rna = rna_runs[0]
!fasterq-dump {test_rna} --split-files --outdir "{base_path}/rna" --threads 4
!gzip "{base_path}/rna/{test_rna}"*.fastq

!ls -lh "{base_path}/dna/"
!ls -lh "{base_path}/rna/"

spots read      : 1,433,760
reads read      : 2,867,520
reads written   : 2,867,520
spots read      : 442,928
reads read      : 885,856
reads written   : 885,856
total 127M
-rw------- 1 root root 64M Apr 20 03:06 SRR16195157_1.fastq.gz
-rw------- 1 root root 63M Apr 20 03:06 SRR16195157_2.fastq.gz
total 40M
-rw------- 1 root root 20M Apr 20 03:07 SRR16195160_1.fastq.gz
-rw------- 1 root root 20M Apr 20 03:07 SRR16195160_2.fastq.gz
